# ✈️ Flight Delay Prediction — Spark ML Pipeline
> **Stack:** ClickHouse (JDBC) → Apache Spark → PySpark ML 
> **Target:** `arr_del15_label` (0 = Tepat Waktu, 1 = Delay ≥15 menit)  
> **Split Strategy:** Time-based (Train: 2024, Test: 2025)  
> **Models:** Logistic Regression · Random Forest · GBTClassifier

## 🎯 Justifikasi Pemilihan Model

Masalah ini diframe sebagai binary classification (delay/tidak) 
per penerbangan, bukan time series forecasting, karena:
- Target prediksi adalah keputusan per-flight, bukan tren agregat
- Fitur tersedia per penerbangan (rute, maskapai, jam, historis delay)

Model dipilih dengan strategi berlapis:
- Logistic Regression → baseline, interpretable
- Random Forest → tangkap non-linearitas, robust terhadap outlier  
- GBT Classifier → ensemble boosting, efektif untuk imbalanced data

Metrik utama: F1-Score & ROC-AUC karena data imbalance (~80:20)

## 📦 1. IMPORT LIBRARY

In [1]:
import os
import sys
import time
import shutil
import numpy as np
import pandas as pd
import clickhouse_connect
import threading
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier,     
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator

print('✅ Semua library berhasil di-import!')

✅ Semua library berhasil di-import!


## 🔥 2. INISIALISASI SPARK

In [2]:
# Bersihkan sesi lama jika masih nyangkut
if 'spark' in locals():
    try: spark.stop()
    except: pass

path_jar = os.path.join("..", "spark", "conf", "clickhouse-jdbc-0.6.3.jar")

if not os.path.exists(path_jar):
    print(f"❌ File JAR tidak ditemukan di '{path_jar}'!")
    print("Pastikan file ada di folder yang sama dengan notebook ini.")
else:
    print("✅ File JAR ditemukan! Memulai SparkSession...")

    python_path = sys.executable
    os.environ["PYSPARK_PYTHON"]        = python_path
    os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

    spark = SparkSession.builder \
        .appName("FlightDelayModelBuilder") \
        .config("spark.jars", path_jar) \
        .config("spark.driver.extraClassPath", path_jar) \
        .config("spark.executor.extraClassPath", path_jar) \
        .config("spark.executor.memory", "8g") \
        .config("spark.driver.memory", "8g") \
        .config("spark.driver.maxResultSize", "4g") \
        .config("spark.memory.offHeap.enabled", "true") \
        .config("spark.memory.offHeap.size", "1g") \
        .config("spark.memory.fraction", "0.7") \
        .config("spark.memory.storageFraction", "0.3") \
        .config("spark.pyspark.python", python_path) \
        .config("spark.pyspark.driver.python", python_path) \
        .config("spark.executorEnv.PYSPARK_PYTHON", python_path) \
        .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC -XX:G1HeapRegionSize=32m") \
        .config("spark.driver.extraJavaOptions", "-XX:+UseG1GC -XX:G1HeapRegionSize=32m") \
        .getOrCreate()

    print("✅ SparkSession berhasil terhubung!")
    print(f"   Spark version: {spark.version}")

✅ File JAR ditemukan! Memulai SparkSession...
✅ SparkSession berhasil terhubung!
   Spark version: 4.1.2


## 📥 3. MEMBACA DATA DARI CLICKHOUSE

In [3]:
# ── Konfigurasi koneksi ClickHouse ───────────────────────────────
CLICKHOUSE_URL = (
    "jdbc:clickhouse://13.215.79.3:8123/flight_delay"
    "?compress=false"
    "&connection_timeout=120000"
    "&socket_timeout=120000"
    "&http_connection_provider=HTTP_URL_CONNECTION"
)

print("📥 Menarik data dari ClickHouse (12 partisi bulan)...")
start_time = time.time()

df_all = spark.read.format("jdbc") \
    .option("url", CLICKHOUSE_URL) \
    .option("dbtable", "ontime_features") \
    .option("user", "default") \
    .option("password", "rahasia123") \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .option("partitionColumn", "flight_month") \
    .option("lowerBound", "1") \
    .option("upperBound", "6") \
    .option("numPartitions", "6") \
    .option("fetchsize", "5000") \
    .load()

print(f"✅ Schema dimuat dalam {time.time()-start_time:.1f} detik")
print(f"   Total kolom: {len(df_all.columns)}")

# ── Alternatif: baca dari Parquet lokal (uncomment jika perlu) ───
# df_all = spark.read.parquet("../data/ontime_features.parquet")

# Cek skema & ketersediaan kolom
print("📋 Skema kolom ontime_features:")
for col_name, dtype in df_all.dtypes:
    print(f"  {col_name}: {dtype}")

📥 Menarik data dari ClickHouse (12 partisi bulan)...
✅ Schema dimuat dalam 7.7 detik
   Total kolom: 44
📋 Skema kolom ontime_features:
  FlightDate: date
  IATA_CODE_Reporting_Airline: string
  Flight_Number_Reporting_Airline: string
  Origin: string
  Dest: string
  CRSDepTime: int
  flight_year: int
  flight_quarter: int
  flight_month: int
  flight_day: int
  day_of_week: int
  is_weekend: int
  season: string
  dep_hour: int
  arr_hour: int
  dep_time_bucket: string
  arr_time_bucket: string
  route: string
  same_state_route: int
  distance_bucket: int
  CRSElapsedTime: double
  Distance: double
  DistanceGroup: int
  OriginAirportID: int
  OriginState: string
  DestAirportID: int
  DestState: string
  route_avg_arr_delay_prev: double
  route_arr_delay_rate_prev: double
  route_cancel_rate_prev: double
  carrier_arr_delay_rate_prev: double
  carrier_cancel_rate_prev: double
  origin_arr_delay_rate_prev: double
  origin_cancel_rate_prev: double
  dest_arr_delay_rate_prev: double
  

## ✂️ 4. TIME-BASED SPLIT & CACHING

In [4]:
# # Definisikan pemisahan dasar langsung ke variabel target
# df_train_raw = df_all.filter(df_all.FlightDate < "2025-01-01")
# df_test_raw = df_all.filter(df_all.FlightDate >= "2025-01-01")

# # Data Training
# print("\n📥 Tahap A: Mengunduh Data Training (2021-2024) ke RAM...")
# start_train = time.time()

# for month in range(1, 13):
#     print(f"   ⏳ [Partisi {month}/12] Menyedot data penerbangan Bulan ke-{month:02d} dari AWS...")
#     # Ubah filter menggunakan 'flight_month'
#     df_train_raw.filter(df_train_raw.flight_month == month).cache().count()

# df_train_raw = df_train_raw.cache()
# train_count = df_train_raw.count()
# print(f"   ✔️ Sukses! {train_count:,} baris Data Training terkunci di RAM. Durasi: {time.time() - start_train:.2f} detik.")

# # Data Testing
# print("\n📥 Tahap B: Mengunduh Data Testing (Tahun 2025) ke RAM...")
# start_test = time.time()

# df_test_raw = df_test_raw.cache()
# test_count = df_test_raw.count()
# print(f"   ✔️ Sukses! {test_count:,} baris Data Testing terkunci di RAM. Durasi: {time.time() - start_test:.2f} detik.")


In [5]:
# client = clickhouse_connect.get_client(
#     host='13.215.79.3',
#     port=8123,
#     username='default',
#     password='rahasia123',
#     database='flight_delay',
# )

# def tarik_per_bulan(tahun_filter, label):
#     """Tarik data per bulan lalu gabungkan — hindari timeout koneksi."""
    
#     if tahun_filter == 'train':
#         bulan_list = [
#             (tahun, bulan)
#             for tahun in range(2023, 2025)   # 2023 s/d 2024
#             for bulan in range(1, 13)         # Januari s/d Desember
#         ]
#     else:
#         # 2025: tarik bulan yang tersedia saja
#         bulan_list = [(2025, m) for m in range(1, 13)]
    
#     chunks = []
#     for tahun, bulan in bulan_list:
#         print(f"   📦 [{label}] Bulan {tahun}-{bulan:02d}...", end=" ", flush=True)
#         t0 = time.time()
#         try:
#             df_chunk = client.query_df(f"""
#                 SELECT * FROM ontime_features
#                 WHERE flight_year = {tahun}
#                   AND flight_month = {bulan}
#             """)
#             if len(df_chunk) > 0:
#                 chunks.append(df_chunk)
#                 print(f"✅ {len(df_chunk):,} baris ({time.time()-t0:.1f}s)")
#             else:
#                 print(f"⚠️ kosong, skip")
#         except Exception as e:
#             print(f"❌ Gagal: {e}")
#             continue  # skip bulan ini, lanjut ke berikutnya
    
#     if not chunks:
#         raise ValueError(f"Tidak ada data berhasil ditarik untuk {label}")
    
#     result = pd.concat(chunks, ignore_index=True)
#     return result


# # ── Tarik Training (2023-2024) ─────────────────────────────────────────
# print("📥 Menarik data Training (2023-2024) per bulan...")
# t0 = time.time()
# df_train_pd = tarik_per_bulan('train', 'Train')
# print(f"✅ Training selesai! {len(df_train_pd):,} baris — {time.time()-t0:.1f} detik\n")

# # ── Tarik Testing (2025) ──────────────────────────────────────────
# print("📥 Menarik data Testing (2025) per bulan...")
# t0 = time.time()
# df_test_pd = tarik_per_bulan('test', 'Test')
# print(f"✅ Testing selesai! {len(df_test_pd):,} baris — {time.time()-t0:.1f} detik\n")



In [6]:
# # ── STEP 1: Simpan hasil tarik ke Parquet lokal ──────────────────
# print("💾 Menyimpan ke Parquet lokal...")

# os.makedirs("../data", exist_ok=True)

# df_train_pd.to_parquet("../data/train_2024.parquet", index=False)
# print(f"   ✅ Training tersimpan: {len(df_train_pd):,} baris")
# del df_train_pd  # bebaskan RAM Pandas

# df_test_pd.to_parquet("../data/test_2025.parquet", index=False)
# print(f"   ✅ Testing tersimpan: {len(df_test_pd):,} baris")
# del df_test_pd   # bebaskan RAM Pandas

# print("✅ Semua data tersimpan ke Parquet!")

In [7]:
# ── STEP 2: Load Parquet ke Spark (RAM Pandas sudah kosong) ──────
print("⚡ Load Parquet → Spark DataFrame...")

df_train_raw = spark.read.parquet("../data/train_2024.parquet").cache()
train_count  = df_train_raw.count()
print(f"✅ Training terkunci di RAM: {train_count:,} baris")

df_test_raw = spark.read.parquet("../data/test_2025.parquet").cache()
test_count  = df_test_raw.count()
print(f"✅ Testing terkunci di RAM: {test_count:,} baris")

print(f"\n🎉 Semua data siap!")
print(f"   Training : {train_count:,} baris")
print(f"   Testing  : {test_count:,} baris")

⚡ Load Parquet → Spark DataFrame...
✅ Training terkunci di RAM: 11,728,023 baris
✅ Testing terkunci di RAM: 4,756,367 baris

🎉 Semua data siap!
   Training : 11,728,023 baris
   Testing  : 4,756,367 baris


## 🧹 5. CLEANING & VALIDASI LABEL

In [8]:
# Jalankan ini dulu untuk tahu nilai apa saja yang ada
df_train_raw.select("arr_del15_label").distinct().show(50)

+---------------+
|arr_del15_label|
+---------------+
|              1|
|              0|
+---------------+



In [9]:
print("🧹 Membersihkan label arr_del15_label...")

def clean_label(df):
    return (
        df
        # Paksa kolom jadi string dulu — tangani tipe apapun
        .withColumn("arr_del15_label", F.col("arr_del15_label").cast("string"))
        # Ganti semua varian \N dan string kosong jadi null
        .withColumn(
            "arr_del15_label",
            F.when(
                F.trim(F.col("arr_del15_label")).isin(["\\N", "\\\\N", "", "NULL", "null", "nan"]),
                F.lit(None)
            ).otherwise(F.trim(F.col("arr_del15_label")))
        )
        # Hapus null
        .filter(F.col("arr_del15_label").isNotNull())
        # Cast dua tahap: string → double → int (aman untuk "0.0" dan "1.0")
        .withColumn(
            "arr_del15_label",
            F.col("arr_del15_label").cast("double").cast("int")
        )
        # Hanya label 0 dan 1
        .filter(F.col("arr_del15_label").isin([0, 1]))
    )

df_train_raw = clean_label(df_train_raw)
df_test_raw  = clean_label(df_test_raw)

# Cek isi label tanpa operasi yang berisiko
train_clean = df_train_raw.count()
test_clean  = df_test_raw.count()

print(f"   Training bersih : {train_clean:,} baris")
print(f"   Testing bersih  : {test_clean:,} baris")

print("\n📊 Distribusi label Training:")
df_train_raw.groupBy("arr_del15_label").count().orderBy("arr_del15_label").show()

print("📊 Distribusi label Testing:")
df_test_raw.groupBy("arr_del15_label").count().orderBy("arr_del15_label").show()

🧹 Membersihkan label arr_del15_label...
   Training bersih : 11,728,023 baris
   Testing bersih  : 4,756,367 baris

📊 Distribusi label Training:
+---------------+-------+
|arr_del15_label|  count|
+---------------+-------+
|              0|9307806|
|              1|2420217|
+---------------+-------+

📊 Distribusi label Testing:
+---------------+-------+
|arr_del15_label|  count|
+---------------+-------+
|              0|3719567|
|              1|1036800|
+---------------+-------+



## ⚖️ 6. HANDLE IMBALANCE (CLASS WEIGHT)

In [10]:
# Hitung bobot berbanding terbalik dengan frekuensi kelas
count_total  = df_train_raw.count()
count_delay  = df_train_raw.filter(F.col("arr_del15_label") == 1).count()
count_ontime = df_train_raw.filter(F.col("arr_del15_label") == 0).count()

weight_delay  = count_total / (2.0 * count_delay)   # kelas minoritas → bobot lebih tinggi
weight_ontime = count_total / (2.0 * count_ontime)  # kelas mayoritas → bobot lebih rendah

print(f"⚖️  Imbalance ratio : {count_ontime/count_delay:.2f}:1")
print(f"   weight_delay   : {weight_delay:.4f}")
print(f"   weight_ontime  : {weight_ontime:.4f}")

# Tambahkan kolom class_weight & time_weight ke training
df_train_raw = df_train_raw \
    .withColumn(
        "class_weight",
        F.when(F.col("arr_del15_label") == 1, weight_delay).otherwise(weight_ontime)
    ) \
    .withColumn(
        "time_weight",
        F.when(F.col("FlightDate") < "2023-01-01", 0.5).otherwise(1.0)
    ) \
    .withColumn("final_weight", F.col("class_weight") * F.col("time_weight"))

df_train_balanced = df_train_raw
print("\n✅ Kolom final_weight berhasil ditambahkan!")

⚖️  Imbalance ratio : 3.85:1
   weight_delay   : 2.4229
   weight_ontime  : 0.6300

✅ Kolom final_weight berhasil ditambahkan!


## ⚙️ 7. FEATURE ENGINEERING & PREPROCESSING

In [ ]:
print("⚙️ Memulai transformasi fitur...")

# ── StringIndexer untuk kategorikal ──────────────────────────────
indexer_season  = StringIndexer(inputCol="season", outputCol="season_index",  handleInvalid="keep")
indexer_airline = StringIndexer(inputCol="IATA_CODE_Reporting_Airline", outputCol="airline_index", handleInvalid="keep")
indexer_origin = StringIndexer(inputCol="Origin", outputCol="origin_index", handleInvalid="keep")
indexer_dest   = StringIndexer(inputCol="Dest",   outputCol="dest_index",   handleInvalid="keep")

fit_season  = indexer_season.fit(df_train_balanced)
fit_airline = indexer_airline.fit(df_train_balanced)
fit_origin = indexer_origin.fit(df_train_balanced)
fit_dest = indexer_dest.fit(df_train_balanced)

def apply_indexers(df):
    df = fit_season.transform(df)
    df = fit_airline.transform(df)
    df = fit_origin.transform(df)
    df = fit_dest.transform(df)
    return df

df_train_indexed = apply_indexers(df_train_balanced)
df_test_indexed  = apply_indexers(df_test_raw)

# ── Cast kolom ke double ──────────────────────────────────────────
KOLOM_NUMERIK = [
    "route_avg_arr_delay_prev", "carrier_arr_delay_rate_prev",
    "flight_month", "dep_hour", "arr_hour",
    "is_weekend", "same_state_route",
    "day_of_week", "CRSElapsedTime", "Distance", "flight_day",
    "route_arr_delay_rate_prev", "route_carrier_arr_delay_rate_prev",
    "origin_arr_delay_rate_prev", "origin_cancel_rate_prev",
    "dest_arr_delay_rate_prev",   "dest_cancel_rate_prev",
    "carrier_cancel_rate_prev",   "distance_bucket",
]
for col in KOLOM_NUMERIK:
    df_train_indexed = df_train_indexed.withColumn(col, F.col(col).cast("double"))
    df_test_indexed  = df_test_indexed.withColumn(col,  F.col(col).cast("double"))

# ── Flag has_route_history (sebelum fillna) ───────────────────────
for df_name in ['train', 'test']:
    df = df_train_indexed if df_name == 'train' else df_test_indexed
    df = df.withColumn(
        "has_route_history",
        F.when(F.col("route_avg_arr_delay_prev").isNull(), 0.0).otherwise(1.0)
    )
    if df_name == 'train': df_train_indexed = df
    else:                  df_test_indexed  = df

# ── Imputasi median (hitung dari train saja) ──────────────────────
KOLOM_MEDIAN = [
    "route_avg_arr_delay_prev", "carrier_arr_delay_rate_prev",
    "route_arr_delay_rate_prev", "route_carrier_arr_delay_rate_prev",
    "origin_arr_delay_rate_prev", "origin_cancel_rate_prev",
    "dest_arr_delay_rate_prev", "dest_cancel_rate_prev",
    "carrier_cancel_rate_prev",
]
medians = {}
for col in KOLOM_MEDIAN:
    medians[col] = df_train_indexed.approxQuantile(col, [0.5], 0.01)[0]
    print(f"  Median {col}: {medians[col]:.4f}")
    df_train_indexed = df_train_indexed.fillna(medians[col], subset=[col])
    df_test_indexed  = df_test_indexed.fillna(medians[col],  subset=[col])

OTHER_COLS = [
    "flight_month", "dep_hour", "arr_hour", "is_weekend",
    "same_state_route", "day_of_week", "CRSElapsedTime",
    "Distance", "flight_day", "OriginAirportID",
    "DestAirportID", "distance_bucket",
]
df_train_indexed = df_train_indexed.fillna(0.0, subset=OTHER_COLS)
df_test_indexed  = df_test_indexed.fillna(0.0,  subset=OTHER_COLS)
print("\n✅ Imputasi selesai!")

# ── Encoding siklikal ─────────────────────────────────────────────
def add_cyclical_features(df):
    return df \
        .withColumn("dep_hour_sin",  F.sin(2 * 3.14159 * F.col("dep_hour")     / 24)) \
        .withColumn("dep_hour_cos",  F.cos(2 * 3.14159 * F.col("dep_hour")     / 24)) \
        .withColumn("arr_hour_sin",  F.sin(2 * 3.14159 * F.col("arr_hour")     / 24)) \
        .withColumn("arr_hour_cos",  F.cos(2 * 3.14159 * F.col("arr_hour")     / 24)) \
        .withColumn("month_sin",     F.sin(2 * 3.14159 * F.col("flight_month") / 12)) \
        .withColumn("month_cos",     F.cos(2 * 3.14159 * F.col("flight_month") / 12)) \
        .withColumn("dow_sin",       F.sin(2 * 3.14159 * F.col("day_of_week")  / 7)) \
        .withColumn("dow_cos",       F.cos(2 * 3.14159 * F.col("day_of_week")  / 7))

df_train_indexed = add_cyclical_features(df_train_indexed)
df_test_indexed  = add_cyclical_features(df_test_indexed)

# ── Fitur interaksi ───────────────────────────────────────────────
def add_interaction_features(df):
    return df \
        .withColumn(
            "route_x_carrier_risk",
            F.col("route_arr_delay_rate_prev") * F.col("carrier_arr_delay_rate_prev")
        ) \
        .withColumn(
            "origin_x_dest_risk",
            F.col("origin_arr_delay_rate_prev") * F.col("dest_arr_delay_rate_prev")
        ) \
        .withColumn(
            "carrier_rate_x_peak_hour",
            F.col("carrier_arr_delay_rate_prev") * F.col("dep_hour_sin")
        )

df_train_indexed = add_interaction_features(df_train_indexed)
df_test_indexed  = add_interaction_features(df_test_indexed)

df_train_indexed.unpersist()
df_test_indexed.unpersist()

print("✅ Feature engineering selesai!")

⚙️ Memulai transformasi fitur...
  Median route_avg_arr_delay_prev: 14.7878
  Median carrier_arr_delay_rate_prev: 0.2051
  Median route_arr_delay_rate_prev: 0.2040
  Median route_carrier_arr_delay_rate_prev: 0.2000
  Median origin_arr_delay_rate_prev: 0.2062
  Median origin_cancel_rate_prev: 0.0121
  Median dest_arr_delay_rate_prev: 0.2070
  Median dest_cancel_rate_prev: 0.0119
  Median carrier_cancel_rate_prev: 0.0122

✅ Imputasi selesai!
✅ Feature engineering selesai!


In [ ]:
# ── Definisi fitur final & VectorAssembler ────────────────────────
FEATURE_COLS = [
    # Temporal (siklikal)
    "dep_hour_sin", "dep_hour_cos",
    "month_sin",    "month_cos",
    "dow_sin",      "dow_cos",
    "flight_day",

    # Rute & maskapai
    "airline_index", "OriginAirportID", "DestAirportID",
    "season_index",

    # Operasional
    "Distance", "CRSElapsedTime", "distance_bucket",
    "has_route_history",

    # Historis (paling penting!)
    "route_avg_arr_delay_prev",
    "route_arr_delay_rate_prev",
    "route_carrier_arr_delay_rate_prev",
    "carrier_arr_delay_rate_prev",
    "carrier_cancel_rate_prev",
    "origin_arr_delay_rate_prev",
    "dest_arr_delay_rate_prev",
    "dest_cancel_rate_prev",
    "route_cancel_rate_prev",

    # Interaksi
    "route_x_carrier_risk",
    "origin_x_dest_risk",
    "carrier_rate_x_peak_hour",
]

assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features", handleInvalid="keep")

df_train_final = assembler.transform(df_train_indexed).select("features", "arr_del15_label", "final_weight").cache()
df_test_final  = assembler.transform(df_test_indexed).select("features", "arr_del15_label").cache()

df_train_raw.unpersist()
df_test_raw.unpersist()
df_train_indexed.unpersist()
df_test_indexed.unpersist()

print(f"✅ VectorAssembler selesai! Total fitur: {len(FEATURE_COLS)}")
print(f"   df_train_final sample:"); df_train_final.limit(3).show()
print(f"   df_test_final sample:"); df_test_final.limit(3).show()

✅ VectorAssembler selesai! Total fitur: 28
   df_train_final sample:
+--------------------+---------------+------------------+
|            features|arr_del15_label|      final_weight|
+--------------------+---------------+------------------+
|[0.99999999999911...|              0|0.6300100689679179|
|[0.96592622692099...|              0|0.6300100689679179|
|[2.65358979379681...|              0|0.6300100689679179|
+--------------------+---------------+------------------+

   df_test_final sample:
+--------------------+---------------+
|            features|arr_del15_label|
+--------------------+---------------+
|[-0.7071044357184...|              0|
|[0.86602628831301...|              0|
|[-0.9659248533161...|              0|
+--------------------+---------------+



## 🔍 7.1. DIAGNOSTIK DATA

In [13]:
print("🔍 Diagnostik sebelum training...")

print("\n📊 Distribusi label Training:")
df_train_raw.groupBy("arr_del15_label").count().orderBy("arr_del15_label").show()

print("📊 Distribusi label Testing:")
df_test_raw.groupBy("arr_del15_label").count().orderBy("arr_del15_label").show()

pos_test = df_test_raw.filter(F.col("arr_del15_label") == 1).count()
neg_test = df_test_raw.filter(F.col("arr_del15_label") == 0).count()
baseline = max(pos_test, neg_test) / (pos_test + neg_test)
print(f"🔢 Positive rate test : {pos_test/(pos_test+neg_test):.4f}")
print(f"🔢 Baseline accuracy  : {baseline:.4f}")

print("\n🧪 Null count pada fitur kunci (training):")
null_check_cols = [
    'route_avg_arr_delay_prev', 'carrier_arr_delay_rate_prev',
    'dep_hour', 'is_weekend', 'season_index', 'distance_bucket',
]
df_train_indexed.select(
    [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in null_check_cols]
).show()

🔍 Diagnostik sebelum training...

📊 Distribusi label Training:
+---------------+-------+
|arr_del15_label|  count|
+---------------+-------+
|              0|9307806|
|              1|2420217|
+---------------+-------+

📊 Distribusi label Testing:
+---------------+-------+
|arr_del15_label|  count|
+---------------+-------+
|              0|3719567|
|              1|1036800|
+---------------+-------+

🔢 Positive rate test : 0.2180
🔢 Baseline accuracy  : 0.7820

🧪 Null count pada fitur kunci (training):
+------------------------+---------------------------+--------+----------+------------+---------------+
|route_avg_arr_delay_prev|carrier_arr_delay_rate_prev|dep_hour|is_weekend|season_index|distance_bucket|
+------------------------+---------------------------+--------+----------+------------+---------------+
|                       0|                          0|       0|         0|           0|              0|
+------------------------+---------------------------+--------+----------+--

## 🤖 8. TRAINING MODEL

In [14]:
print("🤖 Menginisialisasi semua model...")

# Spark MLlib models
spark_models = {
    "Logistic Regression": LogisticRegression(
        featuresCol="features",
        labelCol="arr_del15_label",
        weightCol="final_weight",
        maxIter=100,
    ),
    "Random Forest": RandomForestClassifier(
        featuresCol="features",
        labelCol="arr_del15_label",
        weightCol="final_weight",
        numTrees=75,
        maxDepth=8,
        seed=42,
    ),
    "GBT Classifier": GBTClassifier(
        featuresCol="features",
        labelCol="arr_del15_label",
        weightCol="final_weight",  
        maxIter=75,
        maxDepth=5,
        stepSize=0.1,        
        subsamplingRate=0.8,
        seed=42,
    ),
}

trained_spark_models = {}

# Training LR, RF, GBT
print("\n🚀 Training Logistic Regression, Random Forest, GBT...")
for name, model in spark_models.items():
    print(f"   ⏳ Training {name}...")
    t0 = time.time()
    try:
        trained_spark_models[name] = model.fit(df_train_final)
        print(f"   ✅ {name} selesai dalam {time.time()-t0:.1f} detik")
    except Exception as e:
        print(f"   ❌ {name} gagal: {e}")
        trained_spark_models[name] = None

print("\n🎉 Semua model selesai di-training!")

🤖 Menginisialisasi semua model...

🚀 Training Logistic Regression, Random Forest, GBT...
   ⏳ Training Logistic Regression...
   ✅ Logistic Regression selesai dalam 77.4 detik
   ⏳ Training Random Forest...
   ✅ Random Forest selesai dalam 794.3 detik
   ⏳ Training GBT Classifier...
   ✅ GBT Classifier selesai dalam 711.5 detik

🎉 Semua model selesai di-training!


## 🔎 8.1. INSPEKSI MODEL (FEATURE IMPORTANCE)

In [15]:
print("🔎 Inspeksi koefisien & feature importance...")

# Logistic Regression coefficients
lr_model = trained_spark_models.get("Logistic Regression")
if lr_model:
    print("\n📈 Logistic Regression — Koefisien:")
    coeffs = lr_model.coefficients.toArray()
    for name, coef in sorted(zip(FEATURE_COLS, coeffs), key=lambda x: abs(x[1]), reverse=True):
        print(f"  {name:40s}: {coef:+.6f}")

# Random Forest feature importances
rf_model = trained_spark_models.get("Random Forest")
if rf_model:
    print("\n🌲 Random Forest — Feature Importance (Top 15):")
    importances = rf_model.featureImportances.toArray()
    for name, imp in sorted(zip(FEATURE_COLS, importances), key=lambda x: x[1], reverse=True)[:15]:
        print(f"  {name:40s}: {imp:.6f}")

# GBT feature importances
gbt_model = trained_spark_models.get("GBT Classifier")
if gbt_model:
    print("\n🌳 GBT Classifier — Feature Importance (Top 15):")
    importances = gbt_model.featureImportances.toArray()
    for name, imp in sorted(zip(FEATURE_COLS, importances), key=lambda x: x[1], reverse=True)[:15]:
        print(f"  {name:40s}: {imp:.6f}")

🔎 Inspeksi koefisien & feature importance...

📈 Logistic Regression — Koefisien:
  route_x_carrier_risk                    : -12.961942
  route_carrier_arr_delay_rate_prev       : +5.211341
  route_arr_delay_rate_prev               : +3.500609
  carrier_arr_delay_rate_prev             : +3.180831
  carrier_rate_x_peak_hour                : -1.611113
  carrier_cancel_rate_prev                : -1.247001
  origin_arr_delay_rate_prev              : -0.746052
  dest_cancel_rate_prev                   : -0.583800
  dest_arr_delay_rate_prev                : -0.401393
  arr_hour_sin                            : -0.204034
  origin_x_dest_risk                      : +0.183567
  month_cos                               : -0.129751
  month_sin                               : +0.120614
  arr_hour_cos                            : +0.115573
  season_index                            : -0.105693
  dow_sin                                 : -0.092831
  dep_hour_sin                            : +0.067623


## 📊 9. EVALUASI MODEL

In [16]:
print("🕵️ Evaluasi + Threshold Tuning semua model pada data Testing (2025)...")

evaluator_roc = BinaryClassificationEvaluator(
    labelCol="arr_del15_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)
evaluator_pr = BinaryClassificationEvaluator(
    labelCol="arr_del15_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR",
)

# UDF ekstrak probabilitas kelas positif (didefinisikan sekali di luar loop)
extract_prob = F.udf(lambda v: float(v[1]), "double")

final_eval_rows      = []
threshold_rows       = []

for name, model in trained_spark_models.items():
    if model is None:
        print(f"\n⚠️ Skip {name} — model tidak tersedia")
        continue

    print(f"\n⏳ Memproses: {name}...")

    # ── Cache predictions ────────────────────────────────────────
    predictions = model.transform(df_test_final).cache()

    # ── 1. ROC-AUC & PR-AUC (pakai Spark evaluator) ─────────────
    roc_auc = evaluator_roc.evaluate(predictions)
    pr_auc  = evaluator_pr.evaluate(predictions)

    # ── 2. Confusion Matrix (pakai Spark count) ──────────────────
    pred_labels = predictions.select("arr_del15_label", "prediction")
    cm = pred_labels.groupBy("arr_del15_label", "prediction").count().collect()

    # Parse hasil collect() jadi tp/tn/fp/fn
    cm_dict = {(int(row["arr_del15_label"]), int(row["prediction"])): row["count"] for row in cm}
    tp = cm_dict.get((1, 1), 0)
    tn = cm_dict.get((0, 0), 0)
    fp = cm_dict.get((0, 1), 0)
    fn = cm_dict.get((1, 0), 0)

    total       = tp + tn + fp + fn
    accuracy    = (tp + tn) / total if total > 0 else 0.0
    precision   = tp / (tp + fp)    if (tp + fp) > 0 else 0.0
    recall      = tp / (tp + fn)    if (tp + fn) > 0 else 0.0
    f1          = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    specificity = tn / (tn + fp)    if (tn + fp) > 0 else 0.0

    print(f"   Confusion Matrix → TP={tp:,}  TN={tn:,}  FP={fp:,}  FN={fn:,}")
    print(f"   Accuracy={accuracy:.4f} | Precision={precision:.4f} | Recall={recall:.4f}")
    print(f"   F1={f1:.4f} | ROC-AUC={roc_auc:.4f} | PR-AUC={pr_auc:.4f} | Specificity={specificity:.4f}")

    final_eval_rows.append({
        "Algoritma"  : name,
        "Accuracy"   : f"{accuracy:.4f}",
        "Precision"  : f"{precision:.4f}",
        "Recall"     : f"{recall:.4f}",
        "Specificity": f"{specificity:.4f}",
        "F1-Score"   : f"{f1:.4f}",
        "ROC-AUC"    : f"{roc_auc:.4f}",
        "PR-AUC"     : f"{pr_auc:.4f}",
    })

    # ── 3. Threshold Tuning (toPandas SEKALI, langsung di sini) ──
    df_local = predictions.select(
        "arr_del15_label",
        extract_prob(F.col("probability")).alias("prob_delay")
    ).sample(fraction=0.2, seed=42).toPandas()

    y_true  = df_local["arr_del15_label"].values
    y_proba = df_local["prob_delay"].values

    best_f1, best_thresh, best_prec, best_rec = 0, 0.5, 0, 0
    for thresh in np.arange(0.10, 0.90, 0.01):
        y_pred = (y_proba >= thresh).astype(int)
        tp_t = ((y_pred == 1) & (y_true == 1)).sum()
        fp_t = ((y_pred == 1) & (y_true == 0)).sum()
        fn_t = ((y_pred == 0) & (y_true == 1)).sum()
        prec_t = tp_t / (tp_t + fp_t + 1e-9)
        rec_t  = tp_t / (tp_t + fn_t + 1e-9)
        f1_t   = 2 * prec_t * rec_t / (prec_t + rec_t + 1e-9)
        if f1_t > best_f1:
            best_f1, best_thresh, best_prec, best_rec = f1_t, thresh, prec_t, rec_t

    print(f"   🎯 Threshold optimal: {best_thresh:.2f} → F1={best_f1:.4f} | Precision={best_prec:.4f} | Recall={best_rec:.4f}")

    threshold_rows.append({
        "Algoritma"          : name,
        "Threshold"  : f"{best_thresh:.2f}",
        "F1"       : f"{best_f1:.4f}",
        "Precision": f"{best_prec:.4f}",
        "Recall"   : f"{best_rec:.4f}",
    })

    # ── ✅ Unpersist SEGERA setelah semua pemakaian selesai ───────
    predictions.unpersist()
    del df_local, y_true, y_proba   # bebaskan memori Pandas juga

# ── Tabel ringkasan ───────────────────────────────────────────────
print("\n" + "=" * 95)
print("📊 TABEL EVALUASI AKHIR — APACHE SPARK ML PIPELINE")
print("=" * 95)
print(pd.DataFrame(final_eval_rows).to_string(index=False))
print("=" * 95)

print("\n" + "=" * 75)
print("📊 RINGKASAN THRESHOLD TUNING")
print("=" * 75)
print(pd.DataFrame(threshold_rows).to_string(index=False))

baseline_acc = neg_test / (pos_test + neg_test)
print(f"\n📌 Baseline majority-class accuracy: {baseline_acc:.4f}")

🕵️ Evaluasi + Threshold Tuning semua model pada data Testing (2025)...

⏳ Memproses: Logistic Regression...
   Confusion Matrix → TP=620,608  TN=2,336,324  FP=1,383,243  FN=416,192
   Accuracy=0.6217 | Precision=0.3097 | Recall=0.5986
   F1=0.4082 | ROC-AUC=0.6536 | PR-AUC=0.3263 | Specificity=0.6281
   🎯 Threshold optimal: 0.46 → F1=0.4118 | Precision=0.2917 | Recall=0.6997

⏳ Memproses: Random Forest...
   Confusion Matrix → TP=669,870  TN=2,174,482  FP=1,545,085  FN=366,930
   Accuracy=0.5980 | Precision=0.3024 | Recall=0.6461
   F1=0.4120 | ROC-AUC=0.6567 | PR-AUC=0.3277 | Specificity=0.5846
   🎯 Threshold optimal: 0.47 → F1=0.4127 | Precision=0.2905 | Recall=0.7126

⏳ Memproses: GBT Classifier...
   Confusion Matrix → TP=594,027  TN=2,409,482  FP=1,310,085  FN=442,773
   Accuracy=0.6315 | Precision=0.3120 | Recall=0.5729
   F1=0.4040 | ROC-AUC=0.6565 | PR-AUC=0.3372 | Specificity=0.6478
   🎯 Threshold optimal: 0.46 → F1=0.4103 | Precision=0.2947 | Recall=0.6748

📊 TABEL EVALUASI A

## 🎯 10. THRESHOLD TUNING (OPTIMAL F1)

In [17]:
# print("🎯 Mencari threshold optimal (maksimalkan F1-Score)...")

# # UDF untuk ekstrak probabilitas kelas positif
# extract_prob = F.udf(lambda v: float(v[1]), "double")

# threshold_results = []

# for name, predictions in all_predictions.items():
#     print(f"\n🔍 Model: {name}")

#     # Tarik ke Pandas sekali — jauh lebih efisien dari 60x Spark jobs
#     df_local = predictions.select(
#         "arr_del15_label",
#         extract_prob(F.col("probability")).alias("prob_delay")
#     ).toPandas()

#     y_true  = df_local["arr_del15_label"].values
#     y_proba = df_local["prob_delay"].values

#     best_f1, best_thresh, best_prec, best_rec = 0, 0.5, 0, 0

#     for thresh in np.arange(0.10, 0.90, 0.01):
#         y_pred = (y_proba >= thresh).astype(int)
#         tp = ((y_pred == 1) & (y_true == 1)).sum()
#         fp = ((y_pred == 1) & (y_true == 0)).sum()
#         fn = ((y_pred == 0) & (y_true == 1)).sum()
#         prec = tp / (tp + fp + 1e-9)
#         rec  = tp / (tp + fn + 1e-9)
#         f1   = 2 * prec * rec / (prec + rec + 1e-9)
#         if f1 > best_f1:
#             best_f1, best_thresh, best_prec, best_rec = f1, thresh, prec, rec

#     threshold_results.append({
#         "Algoritma"          : name,
#         "Optimal Threshold"  : f"{best_thresh:.2f}",
#         "F1 @ Optimal"       : f"{best_f1:.4f}",
#         "Precision @ Optimal": f"{best_prec:.4f}",
#         "Recall @ Optimal"   : f"{best_rec:.4f}",
#     })
#     print(f"   Threshold optimal: {best_thresh:.2f}")
#     print(f"   F1={best_f1:.4f} | Precision={best_prec:.4f} | Recall={best_rec:.4f}")

# print("\n" + "=" * 75)
# print("📊 RINGKASAN THRESHOLD TUNING")
# print("=" * 75)
# print(pd.DataFrame(threshold_results).to_string(index=False))

## 💾 11. SIMPAN MODEL & ENCODER

In [18]:
# os.makedirs("../models", exist_ok=True)

# # ── Simpan semua model ────────────────────────────────────────────
# for name, model in trained_spark_models.items():
#     if model is None:
#         print(f"⚠️ Skip {name} — model tidak tersedia")
#         continue

#     save_path = os.path.join("..", "models", name.lower().replace(" ", "_"))
#     if os.path.exists(save_path):
#         shutil.rmtree(save_path)
#     model.save(save_path)
#     print(f"✅ Tersimpan: {save_path}/")

# # ── Simpan encoder ────────────────────────────────────────────────
# for enc_name, encoder in [("encoder_season", fit_season), ("encoder_airline", fit_airline)]:
#     save_path = f"../models/{enc_name}"
#     if os.path.exists(save_path):
#         shutil.rmtree(save_path)
#     encoder.save(save_path)
#     print(f"✅ Tersimpan: {save_path}/")

# # Simpan FEATURE_COLS dan medians sebagai JSON
# import json
# with open("../models/feature_cols.json", "w") as f:
#     json.dump(FEATURE_COLS, f)
# with open("../models/medians.json", "w") as f:
#     json.dump(medians, f)

# print("\n🎉 Semua artifact berhasil disimpan!")
# print("\n# Cara load kembali:")
# print("# from pyspark.ml.classification import LogisticRegressionModel, RandomForestClassificationModel, GBTClassificationModel")
# print("# from pyspark.ml.feature import StringIndexerModel")
# print("# lr_model   = LogisticRegressionModel.load('../models/logistic_regression')")
# print("# rf_model   = RandomForestClassificationModel.load('../models/random_forest')")
# print("# gbt_model  = GBTClassificationModel.load('../models/gbt_classifier')")
# print("# fit_season = StringIndexerModel.load('../models/encoder_season')")